# Occlusion JEPA — entraînement

Un disque rebondit sur un écran 64×64 ; une barre verticale — position aléatoire **par épisode** (fixe au sein d'un épisode) — l'occulte totalement pendant ~3 frames.
On entraîne un **JEPA temporel** (encodeur CNN → prédicteur **markovien sans état** en rollout latent, cible EMA, régularisation VICReg), puis on évalue : probes post-hoc, métriques latentes, rollouts décodés.

Le prédicteur markovien (`ẑ_t = ẑ_{t-1} + MLP(ẑ_{t-2}, ẑ_{t-1})`) n'a **aucun état caché privé** : la seule mémoire qui traverse une occlusion est le latent lui-même. Baseline récurrente disponible via `Config(predictor_type="gru")`.

Pour l'exploration PCA 3D interactive, voir `viz_pca.ipynb`.

> Runtime → GPU (T4 suffisant).

## 1. Setup

In [ ]:
import os, sys

REPO_URL = "https://github.com/<TON_USER>/occlusion-jepa.git"  # ← à remplacer

if not os.path.exists("occlusion-jepa"):
    !git clone {REPO_URL} occlusion-jepa
%pip install -q -r occlusion-jepa/requirements.txt
sys.path.insert(0, "occlusion-jepa/src")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Sanity check des données

Une séquence générée : 4 frames de contexte + 8 d'horizon. Bord **rouge** = disque totalement occlus (attendu : ~3 frames par traversée). La barre change de position d'une séquence à l'autre.

In [ ]:
from occlusion_jepa import Config
from occlusion_jepa.viz import plot_sequence

cfg = Config()  # tous les hyperparamètres sont là (data, modèles, loss, train)
for seed in [3, 7]:
    plot_sequence(cfg, seed=seed)

## 3. Entraînement JEPA

60 epochs × 500 steps (≈20-40 min sur T4). À chaque epoch : éval sur un set de validation figé, sauvegarde de **`last.pt`** (toujours) et **`best.pt`** (quand `val_inv` s'améliore) → téléchargeables *en cours d'entraînement* via le navigateur de fichiers Colab (panneau 📁 à gauche).

Surveiller `z_std` ≈ 1 (si ↓ vers 0 : collapse, la variance VICReg est censée l'empêcher).

In [ ]:
from occlusion_jepa.train import train_jepa
from occlusion_jepa.viz import plot_history

# Option : sauver les checkpoints sur Drive pour survivre aux déconnexions
# from google.colab import drive; drive.mount("/content/drive")
# ckpt_dir = "/content/drive/MyDrive/occlusion-jepa"
ckpt_dir = "."

encoder, ema_encoder, predictor, history = train_jepa(cfg, device=device, ckpt_dir=ckpt_dir)
plot_history(history)

## 4. Probes post-hoc (encodeur gelé)

- **decoder-obs** : ẑ → frame observée (avec barre)
- **decoder-oracle** : ẑ → frame *sans* barre — si le disque décodé apparaît à la bonne position pendant l'occlusion, l'information a survécu en latent
- **probe linéaire** z → (x, y) : le test quantitatif, ventilé visible / occlus, sur embeddings réels et prédits

In [ ]:
from occlusion_jepa.probes import train_decoder, probe_report

decoder_obs = train_decoder(cfg, encoder, device, oracle=False)
decoder_oracle = train_decoder(cfg, encoder, device, oracle=True)

import json
report = probe_report(cfg, encoder, predictor, device)
print(json.dumps(report, indent=2, ensure_ascii=False))

## 5. Métriques latentes du rollout

MSE / cosine entre ẑ prédit et z̄ réel (encodeur EMA), par pas de rollout — les barres rouges indiquent la fraction de frames occluses à ce pas.

In [ ]:
from occlusion_jepa.viz import latent_metrics, plot_latent_metrics

metrics = latent_metrics(cfg, ema_encoder, encoder, predictor, device)
print(metrics["split"])
plot_latent_metrics(metrics)

## 6. Rollouts décodés

Séquences traversant la barre : frames réelles vs frames décodées depuis les ẑ prédits. La ligne **ẑ→oracle** est le verdict visuel : pendant les frames à bord rouge, le disque décodé est-il à la bonne position ?

In [ ]:
from occlusion_jepa.viz import plot_decoded_rollouts

plot_decoded_rollouts(cfg, encoder, predictor, decoder_obs, decoder_oracle, device)

---
### Boutons de réglage (dans `Config`)
- durée d'occlusion : `bar_width`, `disk_radius`, `speed_x_range` — occlusion totale ≈ `(bar_width − 2·r) / |vx|`
- position de la barre : `bar_margin` (plage de tirage par épisode)
- prédicteur : `predictor_type` — `"markov"` (MLP sans état : la permanence DOIT passer par les ẑ) ou `"gru"` (état caché privé, baseline)
- horizon / contexte : `horizon`, `context_len`
- coefficients VICReg : `lambda_inv`, `lambda_var`, `lambda_cov`
- durée d'entraînement : `epochs`, `steps_per_epoch`

### Recharger un checkpoint (ex. le best d'un run précédent)
```python
from occlusion_jepa.train import load_checkpoint
cfg, encoder, ema_encoder, predictor, history = load_checkpoint("best.pt", device=device)
```